# Compana ML — Modul 1: Pretext Analyzer & Problem Classifier

**Teknik:** NLP Klasifikasi Teks  
**Pipeline:** Cleaning → Casefolding → Slang Fix → Tokenizing → Filtering Stopwords → Stemming → TF-IDF → ML Model  
**Model:** Naive Bayes, Logistic Regression, Random Forest  
**Output:** User Context Object (SRS F-02.4)

| Field | Referensi SRS |
|---|---|
| `problem_category` | F-03.1 |
| `target_role` | F-03.2 |
| `current_level` | F-02.3 |
| `intent` | F-02.3 |
| `extracted_keywords` | F-02.3 (TF-IDF based) |
| `confidence_score` | F-03.3 |
| `needs_clarification` | F-03.4 |

> **Save/Load:** joblib — model tidak perlu training ulang setiap run  
> **Ganti Dataset:** Untuk pakai dataset lain, cukup ganti file CSV di folder `data/` dan sesuaikan nama kolom `text` dan `label` jika berbeda.

## 1. Import Library

In [1]:
import re
import os
import string
import warnings
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    _factory  = StemmerFactory()
    _stemmer  = _factory.create_stemmer()
    SASTRAWI  = True
    print("Sastrawi berhasil diload")
except ImportError:
    SASTRAWI  = False
    print("Sastrawi tidak ditemukan! stemming dinonaktifkan (install: pip install PySastrawi)")

Sastrawi berhasil diload


## 2. Load Dataset dari Folder `data/`

In [2]:
DATA_DIR = os.path.join(os.getcwd(), 'data')

def _load_csv(filename: str) -> list:
    path = os.path.join(DATA_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"File tidak ditemukan: {path}")

    df = pd.read_csv(path)

    if 'text' not in df.columns or 'label' not in df.columns:
        raise ValueError(f"{filename} harus punya kolom 'text' dan 'label'")

    return list(zip(df['text'], df['label']))

RAW_DATA    = _load_csv('dataset_problem.csv')
ROLE_DATA   = _load_csv('dataset_role.csv')
LEVEL_DATA  = _load_csv('dataset_level.csv')
INTENT_DATA = _load_csv('dataset_intent.csv')

print(f"dataset_problem.csv  : {len(RAW_DATA)} samples")
print(f"dataset_role.csv     : {len(ROLE_DATA)} samples")
print(f"dataset_level.csv    : {len(LEVEL_DATA)} samples")
print(f"dataset_intent.csv   : {len(INTENT_DATA)} samples")

dataset_problem.csv  : 80 samples
dataset_role.csv     : 42 samples
dataset_level.csv    : 36 samples
dataset_intent.csv   : 36 samples


## 3. Preprocessing Pipeline
Cleaning → Casefolding → Slang Fix → Tokenizing → Filtering → Stemming

In [3]:
# ── Kamus Slang ──────────────────────────────────────────────────────────────
SLANG_WORDS = {
    "gabisa": "tidak bisa", "gapunya": "tidak punya", "gaada": "tidak ada",
    "pengen": "ingin", "pgn": "ingin",
    "bgt": "sangat", "bngt": "sangat",
    "newb": "pemula", "noob": "pemula", "newbie": "pemula",
    "mumet": "bingung", "pusing": "bingung",
    "susah": "sulit", "ribet": "sulit",
    "capek": "lelah", "stress": "tertekan",
    "lost": "tidak arah", "overwhelmed": "kewalahan",
    "udah": "sudah", "udh": "sudah",
    "blm": "belum", "blum": "belum",
    "krn": "karena", "karna": "karena",
    "yg": "yang", "dgn": "dengan", "utk": "untuk",
    "tp": "tetapi", "tapi": "tetapi",
    "nggak": "tidak", "gak": "tidak", "ga": "tidak", "ngga": "tidak",
    "mau": "ingin", "pingin": "ingin",
    "bener": "benar", "beneran": "benar",
    "banget": "sangat", "bgt": "sangat",
    "gimana": "bagaimana", "gmn": "bagaimana",
    "emang": "memang", "emg": "memang",
}

# ── Custom Stopwords ──────────────────────────────────────────────────────────
CUSTOM_STOPWORDS = {
    'dan', 'yang', 'untuk', 'ke', 'dari', 'dengan', 'ini', 'itu',
    'atau', 'jadi', 'ada', 'tidak', 'sudah', 'belum', 'bisa',
    'saya', 'aku', 'kamu', 'dia', 'kami', 'kita', 'mereka',
    'di', 'pada', 'dalam', 'oleh', 'sebagai', 'juga', 'aja',
    'deh', 'sih', 'loh', 'dong', 'ya', 'yah', 'harus', 'perlu',
    'akan', 'tetapi', 'namun', 'karena', 'supaya', 'agar',
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be',
    'have', 'has', 'had', 'do', 'does', 'did', 'to', 'of',
    'mau', 'ingin', 'tahu', 'banyak', 'sangat', 'terlalu',
    'masih', 'sudah', 'pernah', 'punya', 'sama', 'lebih',
}


def cleaning_text(text: str) -> str:
    """Hapus mention, hashtag, URL, angka, dan tanda baca"""
    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    text = re.sub(r'#[A-Za-z0-9]+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.replace('\n', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()


def casefolding_text(text: str) -> str:
    """Ubah semua huruf menjadi lowercase"""
    return text.lower()


def fix_slang(text: str) -> str:
    """Normalisasi kata slang ke kata baku"""
    words = text.split()
    return ' '.join([SLANG_WORDS.get(w.lower(), w) for w in words])


def tokenizing_text(text: str) -> list:
    """Pisah teks menjadi list token"""
    return text.split()


def filtering_text(tokens: list) -> list:
    """Hapus stopwords dan kata pendek (<=2 karakter)"""
    return [w for w in tokens if w not in CUSTOM_STOPWORDS and len(w) > 2]


def stemming_text(tokens: list) -> str:
    """Stemming menggunakan Sastrawi (jika tersedia)"""
    if SASTRAWI:
        return ' '.join([_stemmer.stem(w) for w in tokens])
    return ' '.join(tokens)


def preprocess_pipeline(text: str) -> str:
    """Pipeline lengkap NLP sesuai materi Dicoding"""
    text = cleaning_text(text)
    text = casefolding_text(text)
    text = fix_slang(text)
    tokens = tokenizing_text(text)
    tokens = filtering_text(tokens)
    return stemming_text(tokens)


# ── Demo pipeline ─────────────────────────────────────────────────────────────
sample = "Saya gabisa ngerti gimana cara mulai belajar cybersecurity, bgt overwhelmed"
print(f"Input    : {sample}")
print(f"Output   : {preprocess_pipeline(sample)}")

Input    : Saya gabisa ngerti gimana cara mulai belajar cybersecurity, bgt overwhelmed
Output   : ngerti bagaimana cara mulai ajar cybersecurity kewalahan


## 4. Base ML Model
Reusable class untuk semua classifier berisi TF-IDF + 3 model dengan auto best-model selection.

In [4]:
class BaseNLPClassifier:
    """
    Base class untuk semua NLP classifier di Compana.
    Berisi TF-IDF + 3 model (NB, LR, RF) dengan auto best-model selection.
    """

    def __init__(self, name: str, max_features: int = 150):
        self.name          = name
        self.tfidf         = TfidfVectorizer(max_features=max_features,
                                             min_df=1, max_df=0.95)
        self.label_encoder = LabelEncoder()
        self.models        = {
            'naive_bayes':         BernoulliNB(),
            'logistic_regression': LogisticRegression(max_iter=500, random_state=42),
            'random_forest':       RandomForestClassifier(n_estimators=100, random_state=42),
        }
        self.best_model      = None
        self.best_model_name = None
        self.label_map       = {}
        self.is_trained      = False

    def train(self, data: list, verbose: bool = True) -> dict:
        """
        Train dari list of (text, label) tuples.
        Returns: dict hasil accuracy semua model
        """
        df = pd.DataFrame(data, columns=['text', 'label'])

        if verbose:
            print(f"\n{'='*55}")
            print(f"TRAINING — {self.name}")
            print(f"{'='*55}")
            print(f"Dataset : {len(df)} samples, {df['label'].nunique()} kelas")
            print(df['label'].value_counts().to_string())

        # Preprocessing
        df['processed'] = df['text'].apply(preprocess_pipeline)

        # Encode label
        y = self.label_encoder.fit_transform(df['label'])
        self.label_map = {i: l for i, l in
                          enumerate(self.label_encoder.classes_)}

        # TF-IDF
        X = self.tfidf.fit_transform(df['processed'])
        if verbose:
            print(f"TF-IDF  : {X.shape}")

        # Split 80:20
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42,
            stratify=y if len(df) >= df['label'].nunique() * 2 else None
        )

        # Train semua model
        results   = {}
        best_acc  = 0
        if verbose:
            print(f"\n{'Model':<25} {'Train':>8} {'Test':>8}")
            print("-" * 45)

        for mname, model in self.models.items():
            model.fit(X_train.toarray(), y_train)
            acc_tr = accuracy_score(y_train, model.predict(X_train.toarray()))
            acc_te = accuracy_score(y_test,  model.predict(X_test.toarray()))
            results[mname] = {'train': acc_tr, 'test': acc_te}

            if verbose:
                print(f"{mname:<25} {acc_tr:>7.2%} {acc_te:>7.2%}")

            if acc_te > best_acc:
                best_acc         = acc_te
                self.best_model  = model
                self.best_model_name = mname

        if verbose:
            print(f"\nBest    : {self.best_model_name} (Test: {best_acc:.2%})")

        self.is_trained = True
        return results

    def predict(self, text: str, conf_threshold: float = 0.45) -> dict:
        """
        Prediksi label dari teks.
        Returns: dict {label, confidence, all_probs, low_confidence}
        """
        if not self.is_trained:
            raise RuntimeError(f"{self.name} belum dilatih.")

        processed  = preprocess_pipeline(text)
        X          = self.tfidf.transform([processed])
        pred       = self.best_model.predict(X.toarray())[0]
        proba      = (self.best_model.predict_proba(X.toarray())[0]
                      if hasattr(self.best_model, 'predict_proba') else None)

        confidence = float(max(proba)) if proba is not None else 0.8
        label      = self.label_map[pred]
        all_probs  = ({self.label_map[i]: round(float(p), 3)
                       for i, p in enumerate(proba)}
                      if proba is not None else {})

        return {
            'label':          label,
            'confidence':     round(confidence, 3),
            'all_probs':      all_probs,
            'low_confidence': confidence < conf_threshold,
        }

    def save(self, path: str):
        """Simpan model ke file .pkl"""
        joblib.dump({
            'tfidf':         self.tfidf,
            'label_encoder': self.label_encoder,
            'models':        self.models,
            'best_model':    self.best_model,
            'best_name':     self.best_model_name,
            'label_map':     self.label_map,
        }, path)
        print(f"Model disimpan → {path}")

    def load(self, path: str):
        """Load model dari file .pkl"""
        data                 = joblib.load(path)
        self.tfidf           = data['tfidf']
        self.label_encoder   = data['label_encoder']
        self.models          = data['models']
        self.best_model      = data['best_model']
        self.best_model_name = data['best_name']
        self.label_map       = data['label_map']
        self.is_trained      = True
        print(f"Model dimuat ← {path}")


print("BaseNLPClassifier siap")

BaseNLPClassifier siap


## 5. TF-IDF Keyword Extractor
Ekstrak keyword penting berdasarkan TF-IDF score lebih akurat dari sekedar `split()` karena mempertimbangkan seberapa informatif sebuah kata.

In [5]:
class TFIDFKeywordExtractor:
    """
    Ekstrak keyword penting dari teks berdasarkan TF-IDF score.
    """

    def __init__(self, max_features: int = 200):
        self.tfidf      = TfidfVectorizer(max_features=max_features,
                                          min_df=1, max_df=0.9)
        self.is_fitted  = False

    def fit(self, corpus: list):
        """Fit TF-IDF pada seluruh corpus dataset"""
        processed = [preprocess_pipeline(t) for t in corpus]
        self.tfidf.fit(processed)
        self.is_fitted = True

    def extract(self, text: str, top_n: int = 8) -> list:
        """Ekstrak top-N keyword dari satu teks"""
        if not self.is_fitted:
            # Fallback: simple tokenize
            tokens = filtering_text(tokenizing_text(
                casefolding_text(cleaning_text(text))
            ))
            return list(set(tokens))[:top_n]

        processed  = preprocess_pipeline(text)
        vector     = self.tfidf.transform([processed])
        feature_names = self.tfidf.get_feature_names_out()
        scores     = vector.toarray()[0]

        # Ambil kata dengan skor tertinggi
        top_indices = scores.argsort()[::-1][:top_n]
        keywords    = [feature_names[i] for i in top_indices if scores[i] > 0]
        return keywords


print("TFIDFKeywordExtractor siap")

TFIDFKeywordExtractor siap


## 6. Pretext Analyzer
Orkestrasi semua 4 model ML sekaligus. Output: User Context Object lengkap sesuai SRS F-02.4

In [6]:
class PretextAnalyzer:
    """
    Versi lengkap PretextAnalyzer.
    Mengorkestrasi 4 model ML sekaligus:
      1. ProblemCategoryModel   → beginner_lost / direction_confused / skill_gap / overwhelmed
      2. RoleIdentificationModel → cybersecurity / web_developer / dll
      3. LevelDetectionModel    → beginner / intermediate / advanced
      4. IntentExtractionModel  → learn_new / switch_career / upgrade_skill

    Output: User Context Object lengkap sesuai SRS F-02.4
    """

    MODEL_DIR = 'compana_models'

    def __init__(self):
        self.problem_model = BaseNLPClassifier('Problem Category',   max_features=150)
        self.role_model    = BaseNLPClassifier('Role Identification', max_features=100)
        self.level_model   = BaseNLPClassifier('Level Detection',    max_features=100)
        self.intent_model  = BaseNLPClassifier('Intent Extraction',  max_features=100)
        self.keyword_ext   = TFIDFKeywordExtractor(max_features=200)
        self.is_ready      = False

    def train(self, save: bool = True):
        """
        Train semua 4 model sekaligus.
        save=True → simpan otomatis ke folder compana_models/
        """
        print("=" * 55)
        print("COMPANA ML — MODUL 1: PRETEXT ANALYZER")
        print("Training semua model...")
        print("=" * 55)

        self.problem_model.train(RAW_DATA)
        self.role_model.train(ROLE_DATA)
        self.level_model.train(LEVEL_DATA)
        self.intent_model.train(INTENT_DATA)

        # Fit keyword extractor dengan semua teks gabungan
        all_texts = (
            [t for t, _ in RAW_DATA] +
            [t for t, _ in ROLE_DATA] +
            [t for t, _ in LEVEL_DATA] +
            [t for t, _ in INTENT_DATA]
        )
        self.keyword_ext.fit(all_texts)

        self.is_ready = True

        if save:
            self._save_all()

        print("\nPretextAnalyzer siap digunakan.")

    def _save_all(self):
        """Simpan semua model ke disk"""
        os.makedirs(self.MODEL_DIR, exist_ok=True)
        self.problem_model.save(f'{self.MODEL_DIR}/problem_model.pkl')
        self.role_model.save(f'{self.MODEL_DIR}/role_model.pkl')
        self.level_model.save(f'{self.MODEL_DIR}/level_model.pkl')
        self.intent_model.save(f'{self.MODEL_DIR}/intent_model.pkl')
        joblib.dump(self.keyword_ext, f'{self.MODEL_DIR}/keyword_ext.pkl')
        print(f"Semua model tersimpan di folder '{self.MODEL_DIR}/'")

    def load(self):
        """
        Load semua model dari disk — tidak perlu training ulang.
        Gunakan ini saat production / sudah pernah train sebelumnya.
        """
        self.problem_model.load(f'{self.MODEL_DIR}/problem_model.pkl')
        self.role_model.load(f'{self.MODEL_DIR}/role_model.pkl')
        self.level_model.load(f'{self.MODEL_DIR}/level_model.pkl')
        self.intent_model.load(f'{self.MODEL_DIR}/intent_model.pkl')
        self.keyword_ext = joblib.load(f'{self.MODEL_DIR}/keyword_ext.pkl')
        self.is_ready = True
        print("Semua model berhasil dimuat dari disk.")

    def analyze(self, pretext: str) -> dict:
        """
        Analisis pretext user secara lengkap.

        Returns: User Context Object (SRS F-02.4)
        """
        if not self.is_ready:
            raise RuntimeError("Analyzer belum siap. Panggil .train() atau .load() dulu.")

        problem  = self.problem_model.predict(pretext)
        role     = self.role_model.predict(pretext)
        level    = self.level_model.predict(pretext)
        intent   = self.intent_model.predict(pretext)
        keywords = self.keyword_ext.extract(pretext, top_n=8)

        # Overall confidence = rata-rata semua model
        overall_conf = round(
            (problem['confidence'] + role['confidence'] +
             level['confidence']   + intent['confidence']) / 4, 3
        )

        # Trigger clarification jika salah satu model tidak yakin (SRS F-03.4)
        needs_clarification = (
            problem['low_confidence'] or
            role['low_confidence']    or
            overall_conf < 0.45
        )

        return {
            # ── Core output (SRS F-02.3, F-03.1, F-03.2) ──
            'raw_input':           pretext,
            'problem_category':    problem['label'],
            'problem_confidence':  problem['confidence'],
            'target_role':         role['label'],
            'role_confidence':     role['confidence'],
            'current_level':       level['label'],
            'level_confidence':    level['confidence'],
            'intent':              intent['label'],
            'intent_confidence':   intent['confidence'],
            # ── Keywords berbasis TF-IDF score ─────────────
            'extracted_keywords':  keywords,
            # ── Confidence & clarification (SRS F-03.3/4) ──
            'overall_confidence':  overall_conf,
            'needs_clarification': needs_clarification,
            # ── Detail probabilitas semua kelas ────────────
            'all_probs': {
                'problem': problem['all_probs'],
                'role':    role['all_probs'],
                'level':   level['all_probs'],
                'intent':  intent['all_probs'],
            },
        }


print("PretextAnalyzer siap")

PretextAnalyzer siap


## 7. Training / Load Model
Jika model sudah ada di disk, load saja. Jika belum, training otomatis.

In [7]:
analyzer = PretextAnalyzer()

if not os.path.exists('compana_models/role_model.pkl'):
    print("Model belum ada, mulai training...")
    analyzer.train(save=True)
else:
    print("Model ditemukan di disk, loading...")
    analyzer.load()

Model ditemukan di disk, loading...
Model dimuat ← compana_models/problem_model.pkl
Model dimuat ← compana_models/role_model.pkl
Model dimuat ← compana_models/level_model.pkl
Model dimuat ← compana_models/intent_model.pkl
Semua model berhasil dimuat dari disk.


## 8. Test Cases
Pengujian dengan 6 skenario sesuai SRS.

In [8]:
test_cases = [
    # Skenario utama SRS 2.8.1
    "Saya ingin menjadi cybersecurity tapi benar-benar pemula dan bingung harus mulai dari mana",
    # Direction confused
    "Bingung pilih antara web developer atau data scientist, tidak tahu mana yang lebih bagus untuk karir",
    # Skill gap intermediate
    "Sudah bisa Python dasar dan pernah buat beberapa project tapi skill masih kurang untuk jadi data analyst profesional",
    # Overwhelmed
    "Overwhelmed banget terlalu banyak yang harus dipelajari untuk jadi fullstack developer tidak tahu harus fokus ke mana",
    # Switch career
    "Saya kerja di marketing tapi ingin pindah karir ke bidang IT khususnya mobile developer",
    # Upgrade skill
    "Sudah jadi junior developer dua tahun dan ingin upgrade skill untuk naik ke posisi senior",
]

print("=" * 65)
print("TEST CASES — COMPANA PRETEXT ANALYZER")
print("=" * 65)

for i, text in enumerate(test_cases, 1):
    print(f"\n[Test {i}]")
    print(f"Input   : {text[:70]}...")
    result = analyzer.analyze(text)
    print(f"Category: {result['problem_category']:<25} ({result['problem_confidence']:.0%})")
    print(f"Role    : {result['target_role']:<25} ({result['role_confidence']:.0%})")
    print(f"Level   : {result['current_level']:<25} ({result['level_confidence']:.0%})")
    print(f"Intent  : {result['intent']:<25} ({result['intent_confidence']:.0%})")
    print(f"Conf    : {result['overall_confidence']:.0%}  |  Clarif? {result['needs_clarification']}")
    print(f"Keywords: {', '.join(result['extracted_keywords'][:5])}")

TEST CASES — COMPANA PRETEXT ANALYZER

[Test 1]
Input   : Saya ingin menjadi cybersecurity tapi benar-benar pemula dan bingung h...
Category: direction_confused        (43%)
Role    : cybersecurity             (34%)
Level   : advanced                  (55%)
Intent  : learn_new                 (87%)
Conf    : 55%  |  Clarif? True
Keywords: cybersecurity, mulai, mana, bingung

[Test 2]
Input   : Bingung pilih antara web developer atau data scientist, tidak tahu man...
Category: direction_confused        (72%)
Role    : web_developer             (33%)
Level   : advanced                  (87%)
Intent  : switch_career             (78%)
Conf    : 67%  |  Clarif? True
Keywords: scientist, bagus, antara, pilih, web

[Test 3]
Input   : Sudah bisa Python dasar dan pernah buat beberapa project tapi skill ma...
Category: skill_gap                 (51%)
Role    : data_analyst              (37%)
Level   : intermediate              (68%)
Intent  : upgrade_skill             (63%)
Conf    : 55%  |  Cla

## 9. Demo: Load dari Disk (Tanpa Training Ulang)
Simulasi penggunaan di production model dimuat dari disk, langsung bisa prediksi.

In [9]:
print("=" * 55)
print("DEMO: Load model dari disk (tanpa training ulang)")
print("=" * 55)

analyzer2 = PretextAnalyzer()
analyzer2.load()

result = analyzer2.analyze("Ingin belajar flutter untuk mobile development tapi masih pemula")

print(f"\nHasil analisis:")
print(f"  Category : {result['problem_category']} ({result['problem_confidence']:.0%})")
print(f"  Role     : {result['target_role']} ({result['role_confidence']:.0%})")
print(f"  Level    : {result['current_level']} ({result['level_confidence']:.0%})")
print(f"  Intent   : {result['intent']} ({result['intent_confidence']:.0%})")
print(f"  Keywords : {', '.join(result['extracted_keywords'][:5])}")
print(f"  Clarif?  : {result['needs_clarification']}")

DEMO: Load model dari disk (tanpa training ulang)
Model dimuat ← compana_models/problem_model.pkl
Model dimuat ← compana_models/role_model.pkl
Model dimuat ← compana_models/level_model.pkl
Model dimuat ← compana_models/intent_model.pkl
Semua model berhasil dimuat dari disk.

Hasil analisis:
  Category : skill_gap (28%)
  Role     : mobile_developer (29%)
  Level    : advanced (79%)
  Intent   : learn_new (57%)
  Keywords : development, mobile
  Clarif?  : True


## 10. Inspect Detail Probabilitas
Lihat distribusi probabilitas semua kelas untuk satu input.

In [10]:
custom_input = "Saya ingin menjadi data scientist tapi masih bingung harus mulai dari mana"

result = analyzer.analyze(custom_input)

print(f"Input: {custom_input}\n")

for category, probs in result['all_probs'].items():
    print(f"── {category.upper()} ──")
    if probs:
        for label, prob in sorted(probs.items(), key=lambda x: -x[1]):
            bar = '█' * int(prob * 20)
            print(f"  {label:<30} {prob:.3f}  {bar}")
    print()

Input: Saya ingin menjadi data scientist tapi masih bingung harus mulai dari mana

── PROBLEM ──
  direction_confused             0.457  █████████
  beginner_lost                  0.254  █████
  skill_gap                      0.145  ██
  overwhelmed                    0.144  ██

── ROLE ──
  data_analyst                   0.503  ██████████
  cybersecurity                  0.107  ██
  web_developer                  0.104  ██
  ui_ux_designer                 0.102  ██
  devops                         0.099  █
  mobile_developer               0.085  █

── LEVEL ──
  advanced                       0.526  ██████████
  beginner                       0.411  ████████
  intermediate                   0.063  █

── INTENT ──
  learn_new                      0.802  ████████████████
  upgrade_skill                  0.138  ██
  switch_career                  0.060  █

